**Run the Merged.py**

This is the jupyter notebook to run the hand tracking and grouping algorithm among others. A key aspect is the modification of the data based on this grouping (so the using of the weighted average for the handedness label and score). Another is the 'deletion' of data/ or to be more precise of particular hand data, that has low confidence score of having a correctly identified hand. The third is the usage of a join and discard function to remove noise furtherly. It acts upon the two hands of same handedness cases in the processed data (processed by the two steps before). Its modification of data is a bit more complication, as one needs to make a case differentiation. This is explained in my report of my erasmus internship. If not available to you, I can always send it via: alex.s.kreibich@gmail.com

The first step is not related to how the algorithm works or running it. 
**It is about what to run the algorithm upon!**
To do this, one inputs any directory, that contains the video data (regardless of Tobii or GoPro videos) from the Smartflat study. This directory should ideally follow the same structure as the one introduced by Sam Perochon for running smoothly.
Then the following cells extract all the cuisine and lego video data, that we would wnat to run the algorithm upon and do just that.

Note: video data in the way of: .json files, so the hand landmarks data, where the hand landmarks were previously extracted from the "real" videos using the mediapipe algorithm 

In [ ]:
import os

#Change this: to run it in a different directory.
directory = '/home/perochon/data-gold-final/cuisine'

#Check if the directory exists
if os.path.exists(directory):
    # List all items in the directory
    items = os.listdir(directory)

    #Filter out only the directories
    folders = [item for item in items if os.path.isdir(os.path.join(directory, item))]

    #Print each folder
    for folder in folders:
        print(folder)
else:
    print(f"The directory {directory} does not exist.")


In [ ]:
import os

# Define the directory to list folders
directory = '/home/perochon/data-gold-final/cuisine'

# Check if the directory exists
if os.path.exists(directory):
    # List all items in the directory
    items = os.listdir(directory)

    # Filter out only the directories that start with 'G'
    folders = [item for item in items if os.path.isdir(os.path.join(directory, item)) and item.startswith('G')]

    # Print each folder that starts with 'G'
    for folder in folders:
        print(folder)
else:
    print(f"The directory {directory} does not exist.")


In [ ]:
import os

# Define the directory to list folders
directory = '/home/perochon/data-gold-final/cuisine'
listdir = []
# Check if the directory exists
if os.path.exists(directory):
    # List all items in the directory
    items = os.listdir(directory)

    # Filter out only the directories that start with 'G' and have a subfolder 'Tobii'
    folders = [item for item in items if os.path.isdir(os.path.join(directory, item)) and item.startswith('G') and os.path.isdir(os.path.join(directory, item, 'Tobii'))]

    # Check for files in the 'Tobii' subfolder
    for folder in folders:
        tobii_path = os.path.join(directory, folder, 'Tobii')
        files = os.listdir(tobii_path)
        json_files = [file for file in files if file.startswith('hand_landmarks_mediapipe') and file.endswith('.json')]
        
        # Print the full paths of the JSON files
        for json_file in json_files:
            listdir.append(os.path.join(tobii_path, json_file))
else:
    print(f"The directory {directory} does not exist.")
print(listdir)


For now: we will in the next cell input a path so we only run it for one video. This is for testing purposes.

In [ ]:
#listdir = listdir[0:1]
listdir = ['/home/perochon/data-gold-final/cuisine/G117_P100_LERVer_14062023/Tobii/hand_landmarks_mediapipe_2xJABGjFqNlVawfeghNSVA==.json']


In [ ]:
print('hallo')

In [ ]:

import sys
# Add the directory containing InitialPlots.py to the Python path
#sys.path.append('/home/kreibich/internship_repository/smartflat')
sys.path.append('/home/perochon/smartflat')



In [ ]:
#FPS value (can be adjusted later, or extracted! (todo..))
fps = 25.015333



In [ ]:
from  api.features.hands_processing.main import initial_TwoSame_Handedness_InFrames, initial_TwoSame_Handedness_InSeconds, plot_Initial_Score_Histogram, DetectionAlgFin, process1_TwoSame_Handedness_InFrames, process1_TwoSame_Handedness_InSeconds, process2_TwoSame_Handedness_InFrames, process2_TwoSame_Handedness_InSeconds, plot_score_threshold, plot_score_threshold_empty_frames, plot_hand_detection_with_join_discard, plot_hand_detection2, plot_hand_detection3, openjson


Ideally: **don't rerun the next cell after a first run...**
It will just take a bit of your time. In principle, after running this once, the time consuming part of my code (grouping/ tracking the hands) is done and saved to a file. So then we can just rerun the below cells.


In [ ]:
import json
import os
# Function to open and read JSON data
def openjson(path):
    with open(path, 'r') as f:
        return json.load(f)
save_dir = "/home/perochon/data-gold-final"

for pathHandLandmark in listdir:
    try:
        # Open JSON data
        dataHandLandmark = openjson(pathHandLandmark)
        
        # Extract the relevant folder names from the path
        first_folder = os.path.basename(os.path.dirname(pathHandLandmark))  # Tobii
        second_folder = os.path.basename(os.path.dirname(os.path.dirname(pathHandLandmark)))  # G102_P87_AUXCyr_09122022
        third_folder = os.path.basename(os.path.dirname(os.path.dirname(os.path.dirname(pathHandLandmark))))  # cuisine
        
        # Create the new folder path maintaining the last three levels
        new_folder_path = os.path.join(save_dir, third_folder, second_folder, first_folder)
        os.makedirs(new_folder_path, exist_ok=True)
        
        # Create SaveProcessedData directory within the new folder path
        save_processed_data_dir = os.path.join(new_folder_path, 'hand_landmarks_outputs')
        os.makedirs(save_processed_data_dir, exist_ok=True)
        
        print(f"Processing path: {pathHandLandmark}")
        print(f"New folder path: {new_folder_path}")
        print(f"SaveProcessedData path: {save_processed_data_dir}")
        
        # Run the specified functions
        initial_TwoSame_Handedness_InSeconds(dataHandLandmark, new_folder_path, fps)
        initial_TwoSame_Handedness_InFrames(dataHandLandmark, new_folder_path)
        plot_Initial_Score_Histogram(dataHandLandmark, new_folder_path)
        DetectionAlgFin(dataHandLandmark, pathHandLandmark, save_dir)
    
    except FileNotFoundError as fnf_error:
        print(f"Error processing file: {pathHandLandmark}")
        print(f"File not found error: {fnf_error}")
    except Exception as e:
        print(f"Error processing file: {pathHandLandmark}")
        print(f"Error: {e}")


In [ ]:
import os
import json
save_dir = "/home/perochon/data-gold-final"
os.makedirs(save_dir, exist_ok=True)
# Function to open and read JSON data
def openjson(path):
    with open(path, 'r') as f:
        return json.load(f)


# Iterate over each path in the list
for pathHandLandmark in listdir:
    try:
        # Extract the relevant folder names from the path
        first_folder = os.path.basename(os.path.dirname(pathHandLandmark))  # Tobii
        second_folder = os.path.basename(os.path.dirname(os.path.dirname(pathHandLandmark)))  # G102_P87_AUXCyr_09122022
        third_folder = os.path.basename(os.path.dirname(os.path.dirname(os.path.dirname(pathHandLandmark))))  # Cuisine
        
        # Create the new folder path maintaining the last three levels
        new_folder_path = os.path.join(save_dir, third_folder, second_folder, first_folder)
        os.makedirs(new_folder_path, exist_ok=True)
        
        print(f"Processing path: {pathHandLandmark}")
        print(f"New folder path: {new_folder_path}")
        
        # Define the file path for loading previously saved JSON data
        file_path = os.path.join(new_folder_path, 'hand_landmarks_outputs/SaveProcessedData.json')

        # Load the JSON file
        with open(file_path, 'r') as file:
            loaded_data = json.load(file)

        # Run the additional functions on the loaded data
        process1_TwoSame_Handedness_InFrames(loaded_data, new_folder_path)
        process1_TwoSame_Handedness_InSeconds(loaded_data, new_folder_path, fps)

    except Exception as e:
        print(f"Error loading or processing file: {file_path}")
        print(f"Error: {e}")
        # Continue with the next file
        continue


In [ ]:
import os
import json
#from hand_detection_processing import plot_hand_detection_with_join_discard
import sys
import pandas as pd

# Replace the path with your actual path
sys.path.insert(0, '/home/kreibich/internship_repository/smartflat')

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 5000)

%load_ext autoreload
%autoreload 2

from api.utils.utils_io import get_data_root
from api.datasets.loader import get_dataset

from api.datasets.loader import get_dataset

dset = get_dataset(dataset_name="base", scenario="all")
# Function to open and read JSON data
def openjson(path):
    with open(path, 'r') as f:
        return json.load(f)


# Base directory where new folders will be created
save_dir = "/home/perochon/data-gold-final"
os.makedirs(save_dir, exist_ok=True)

# Iterate over each path in the list
for pathHandLandmark in listdir:
    try:
        # Extract the relevant folder names from the path
        first_folder = os.path.basename(os.path.dirname(pathHandLandmark))  # Tobii
        second_folder = os.path.basename(os.path.dirname(os.path.dirname(pathHandLandmark)))  # G102_P87_AUXCyr_09122022
        third_folder = os.path.basename(os.path.dirname(os.path.dirname(os.path.dirname(pathHandLandmark))))  # Cuisine
        
        # Create the new folder path maintaining the last three levels
        new_folder_path = os.path.join(save_dir, third_folder, second_folder, first_folder)
        os.makedirs(new_folder_path, exist_ok=True)
        
        print(f"Processing path: {pathHandLandmark}")
        print(f"New folder path: {new_folder_path}")
        
        # Define the file path for loading previously saved JSON data
        file_path = os.path.join(new_folder_path, 'hand_landmarks_outputs/SaveProcessedData.json')

        # Load the JSON file
        with open(file_path, 'r') as file:
            loaded_data = json.load(file)

        # Run the additional functions on the loaded data
        process1_TwoSame_Handedness_InFrames(loaded_data, new_folder_path)
        process1_TwoSame_Handedness_InSeconds(loaded_data, new_folder_path, fps)
        process2_TwoSame_Handedness_InFrames(loaded_data,new_folder_path)
        process2_TwoSame_Handedness_InSeconds(loaded_data,new_folder_path, fps)     
        plot_score_threshold(loaded_data,new_folder_path)
        plot_score_threshold_empty_frames(loaded_data, new_folder_path)

        #now we want to recompute the length based on time:
        deltaTimeJoin = 0.5 #seconds
        deltaTimeDiscard = 0.5 #seconds

        rowNumber = dset.metadata.loc[dset.metadata['hand_landmarks_path'] == pathHandLandmark]
        print(rowNumber)

        if not rowNumber.empty:
            fps = rowNumber['fps'].values[0]  # Get the fps value from the first matching row

        else:
            print(f"No matching fps found for {pathHandLandmark}")
            continue
        
        join_len = deltaTimeJoin*fps
        discard_len = deltaTimeDiscard*fps
        
        print("the join and discard lengths are:", join_len,discard_len)

        

        # Run the new function and update the loaded data
        modified_data = plot_hand_detection_with_join_discard(loaded_data, join_len=join_len, discard_len=discard_len, new_folder_path=new_folder_path)
        
       
    except Exception as e:
        print(f"Error loading or processing file: {file_path}")
        print(f"Error: {e}")
        # Continue with the next file
        continue


In [ ]:
import os
import json

# Base directory where new folders will be created
save_dir = "/home/perochon/data-gold-final"
os.makedirs(save_dir, exist_ok=True)



# Iterate over each path in the list
for pathHandLandmark in listdir:
    try:
        # Remove the last file and replace it with "modified_dataJoinAndDiscard.json"
        base_dir = os.path.dirname(pathHandLandmark)
        modified_file_path = os.path.join(base_dir, 'modified_dataJoinAndDiscard.json')

        # Extract the relevant folder names from the path
        first_folder = os.path.basename(base_dir)  # Tobii
        second_folder = os.path.basename(os.path.dirname(base_dir))  # G102_P87_AUXCyr_09122022
        third_folder = os.path.basename(os.path.dirname(os.path.dirname(base_dir)))  # Cuisine
        
        # Create the new folder path maintaining the last three levels
        new_folder_path = os.path.join(save_dir, third_folder, second_folder, first_folder)
        os.makedirs(new_folder_path, exist_ok=True)
        
        print(f"Processing path: {modified_file_path}")
        print(f"New folder path: {new_folder_path}")
        
        # Load the JSON file
        dataHandLandmark = openjson(modified_file_path)
        dataHandLandmark2 = dataHandLandmark  # Use the entire dataset

        # Run the plot_hand_detection2 function
        plot_hand_detection2(dataHandLandmark2, new_folder_path)
        
        # Run the plot_hand_detection3 function
        plot_hand_detection3(dataHandLandmark2, new_folder_path)

    except Exception as e:
        print(f"Error loading or processing file: {modified_file_path}")
        print(f"Error: {e}")
        # Continue with the next file
        continue


In [ ]:
import os
import json
import pandas as pd
from filelock import FileLock

def openjson(pathtofile):
    with open(pathtofile, 'r') as file:
        data = json.load(file)
    return data

def SummaryStatistics(dataHandLandmark2, summary_file_path, pathHandLandmark):
    positionsGlobal = {
        'leftHand': [],
        'rightHand': []
    }

    # Tracking of cases
    multiple_left_detected_frames = {
        2: [],
        3: [],
        4: []
    }
    multiple_right_detected_frames = {
        2: [],
        3: []
    }
    mixed_detected_frames = {
        '3L_1R': [],
        '2L_1R': [],
        '3R_1L': [],
        '2R_1L': [],
        '2R_2L': []
    }
    no_hands_detected_frames = []

    count_no_hands = 0
    count_one_left = 0
    count_one_right = 0
    count_both_hands = 0
    count_left_hands = [0, 0, 0, 0]  # For 1, 2, 3, 4 left hands
    count_right_hands = [0, 0, 0, 0]  # For 1, 2, 3, 4 right hands

    # Tracking low score frames
    low_score_count_one_left = 0
    low_score_count_one_right = 0
    low_score_count_both_hands = 0
    low_score_count_left_hands = [0, 0, 0, 0]
    low_score_count_right_hands = [0, 0, 0, 0]

    for frame_index, item in enumerate(dataHandLandmark2):
        hand_landmarks = item.get('hand_world_landmarks', [])
        handedness_info = item.get('handedness', [])

        left_hand_found = False
        right_hand_found = False

        # Variables to detect multiple hands
        left_count = 0
        right_count = 0

        low_score_frame = False

        for index, hand in enumerate(hand_landmarks):
            if index < len(handedness_info) and handedness_info[index]:
                hand_type_info = handedness_info[index]
                if hand_type_info['score'] > 0.9:  # Only consider hands with score over 0.9
                    hand_type = hand_type_info.get('display_name', 'unknown').lower()
                    if 'left' in hand_type:
                        left_count += 1
                        if not left_hand_found:
                            left_hand_found = True
                            positionsGlobal['leftHand'].append([(landmark['x'], landmark['y'], landmark['z']) for landmark in hand])
                    elif 'right' in hand_type:
                        right_count += 1
                        if not right_hand_found:
                            right_hand_found = True
                            positionsGlobal['rightHand'].append([(landmark['x'], landmark['y'], landmark['z']) for landmark in hand])
                else:
                    low_score_frame = True

        if left_count > 1:
            if left_count in multiple_left_detected_frames:
                multiple_left_detected_frames[left_count].append(frame_index)
            count_left_hands[left_count - 1] += 1
            if low_score_frame:
                low_score_count_left_hands[left_count - 1] += 1
        if right_count > 1:
            if right_count in multiple_right_detected_frames:
                multiple_right_detected_frames[right_count].append(frame_index)
            count_right_hands[right_count - 1] += 1
            if low_score_frame:
                low_score_count_right_hands[right_count - 1] += 1

        if left_count == 3 and right_count == 1:
            mixed_detected_frames['3L_1R'].append(frame_index)
        elif left_count == 2 and right_count == 1:
            mixed_detected_frames['2L_1R'].append(frame_index)
        elif right_count == 3 and left_count == 1:
            mixed_detected_frames['3R_1L'].append(frame_index)
        elif right_count == 2 and left_count == 1:
            mixed_detected_frames['2R_1L'].append(frame_index)
        elif right_count == 2 and left_count == 2:
            mixed_detected_frames['2R_2L'].append(frame_index)

        if left_count == 0 and right_count == 0:
            count_no_hands += 1
            no_hands_detected_frames.append(frame_index)
        elif left_count == 1 and right_count == 0:
            count_one_left += 1
            if low_score_frame:
                low_score_count_one_left += 1
        elif left_count == 0 and right_count == 1:
            count_one_right += 1
            if low_score_frame:
                low_score_count_one_right += 1
        elif left_count == 1 and right_count == 1:
            count_both_hands += 1
            if low_score_frame:
                low_score_count_both_hands += 1

    summary_stats = {
        "count_no_hands": count_no_hands,
        "count_one_left": count_one_left,
        "count_one_right": count_one_right,
        "count_both_hands": count_both_hands,
        "count_two_left_hands": count_left_hands[1],
        "count_three_left_hands": count_left_hands[2],
        "count_four_left_hands": count_left_hands[3],
        "count_two_right_hands": count_right_hands[1],
        "count_three_right_hands": count_right_hands[2],
        "count_four_right_hands": count_right_hands[3],
        "low_score_count_one_left": low_score_count_one_left,
        "low_score_count_one_right": low_score_count_one_right,
        "low_score_count_both_hands": low_score_count_both_hands,
        "low_score_count_two_left_hands": low_score_count_left_hands[1],
        "low_score_count_three_left_hands": low_score_count_left_hands[2],
        "low_score_count_four_left_hands": low_score_count_left_hands[3],
        "low_score_count_two_right_hands": low_score_count_right_hands[1],
        "low_score_count_three_right_hands": low_score_count_right_hands[2],
        "low_score_count_four_right_hands": low_score_count_right_hands[3],
        "file_path": pathHandLandmark,
        "processedData": dataHandLandmark2
    }

    # Convert the summary statistics to a DataFrame
    df = pd.DataFrame([summary_stats])
    df['filePath'] = summary_file_path

    # Define the file path for the CSV file
    lock_file_path = summary_file_path + '.lock'

    # Save the DataFrame to a CSV file
    with FileLock(lock_file_path):
        # Check if the file exists and if it is empty
        file_exists = os.path.isfile(summary_file_path)
        is_file_empty = os.path.getsize(summary_file_path) == 0 if file_exists else True

        if not file_exists or is_file_empty:
            df.to_csv(summary_file_path, index=False)
        else:
            df.to_csv(summary_file_path, mode='a', header=False, index=False)

    print("Summary statistics have been saved to", summary_file_path)

# Base directory where new folders will be created
save_dir = "/home/kreibich/data"
summary_file_path = os.path.join(save_dir, 'Cuisine', 'StatisticsPostProcessing9.csv')
os.makedirs(os.path.dirname(summary_file_path), exist_ok=True)

# Iterate over each path in the list
for pathHandLandmark in listdir:
    try:
        # Remove the last file and replace it with "modified_dataJoinAndDiscard.json"
        base_dir = os.path.dirname(pathHandLandmark)
        modified_file_path = os.path.join(base_dir, 'modified_dataJoinAndDiscard.json')

        # Extract the relevant folder names from the path
        first_folder = os.path.basename(base_dir)  # Tobii
        second_folder = os.path.basename(os.path.dirname(base_dir))  # G102_P87_AUXCyr_09122022
        third_folder = os.path.basename(os.path.dirname(os.path.dirname(base_dir)))  # Cuisine
        
        # Create the new folder path maintaining the last three levels
        new_folder_path = os.path.join(save_dir, third_folder, second_folder, first_folder)
        os.makedirs(new_folder_path, exist_ok=True)
        
        print(f"Processing path: {modified_file_path}")
        print(f"New folder path: {new_folder_path}")
        
        # Load the JSON file
        dataHandLandmark = openjson(modified_file_path)
        dataHandLandmark2 = dataHandLandmark  # Use the entire dataset

        # Run the SummaryStatistics function
        SummaryStatistics(dataHandLandmark2, summary_file_path, pathHandLandmark)

    except Exception as e:
        print(f"Error loading or processing file: {modified_file_path}")
        print(f"Error: {e}")
        # Continue with the next file
        continue


In [ ]:
print('a')


In [ ]:
import pandas as pd
import os

# Define the path to the summary statistics CSV file
summary_file_path = "/home/kreibich/data/Cuisine/StatisticsPostProcessing8.csv"

# Check if the file exists
if os.path.isfile(summary_file_path):
    # Load the DataFrame from the CSV file
    df = pd.read_csv(summary_file_path)
    #print(df)  # Print the DataFrame
else:
    #print(f"File {summary_file_path} does not exist.")
    df = None

df


In [ ]:
df.drop_duplicates(inplace=True)

# Sort the DataFrame based on 'count_one_right' column in ascending order
df.sort_values('count_one_right', ascending=True, inplace=True)

# If you want to reset the index after sorting
df.reset_index(drop=True, inplace=True)


In [ ]:
df['difference'] = ((df['count_one_right'] - df['count_one_left'])) / (0.5 * (df['count_one_right'] + df['count_one_left']))


In [ ]:
df.head()


In [ ]:
df['right>left'] = (df['count_one_right'] > df['count_one_left']).astype(int)


In [ ]:
df


In [ ]:
if df is not None:
    df = df.drop_duplicates()
    plt.scatter(df.index, df['difference'])
    plt.xlabel('Index')
    plt.ylabel('Difference')
    plt.title('Difference vs Index')
    plt.show()


In [ ]:
df['totalNonEmptyFrames'] =(
    df['count_one_left'] +
    df['count_one_right'] +
    df['count_both_hands'] +
    df['count_two_left_hands'] +
    df['count_three_left_hands'] +
    df['count_four_left_hands'] +
    df['count_two_right_hands'] +
    df['count_three_right_hands'] +
    df['count_four_right_hands']
)


In [ ]:
df


In [ ]:
import matplotlib.pyplot as plt

# Assuming df is your DataFrame and it contains the column "ratioNoHandsToAll"
plt.figure(figsize=(10, 6))
plt.hist(df["ratioNoHandsToAll"], bins=50, edgecolor='black', alpha=0.7)
plt.title('Distribution of the ratio of the number of emtpy frames compared to the total number of frames ')

plt.ylabel('Frequency')
plt.grid(True)
plt.show()
